In [ ]:
#classifying_data_as_anemic_nonanemic
import pandas as pd
import os
import shutil

# -----------------------------
# CONFIG
# -----------------------------
csv_path = "/home/khushi/Pixonate/Vit_conjuctiva/image_hb.csv"   # CSV with image_name and hb_value
eye_crop_dir = "eye_crops"               # Folder with YOLO-cropped eye images
output_dir = "final_dataset"             # Where filtered dataset will be saved

# -----------------------------
# HELPER FUNCTION: Hb -> label
# -----------------------------
def get_label(hb):
    if hb < 10:
        return "anemia"
    elif hb > 13:
        return "no_anemia"
    else:
        return "borderline"   # 10 <= Hb <= 13

# -----------------------------
# LOAD CSV
# -----------------------------
df = pd.read_csv(csv_path)

# -----------------------------
# APPLY LABELING
# -----------------------------
df['label'] = df['hb_value'].apply(get_label)

# -----------------------------
# CREATE CLASS FOLDERS
# -----------------------------
classes = df['label'].unique()
for cls in classes:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

# -----------------------------
# COPY IMAGES TO CLASS FOLDERS
# -----------------------------
for _, row in df.iterrows():
    img_name = row['image_name']
    label = row['label']
    
    src_path = os.path.join(eye_crop_dir, img_name)
    dst_path = os.path.join(output_dir, label, img_name)
    
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
    else:
        print(f"⚠️ Image not found: {img_name}")

# -----------------------------
# PRINT DATASET COUNTS
# -----------------------------
for cls in classes:
    cls_path = os.path.join(output_dir, cls)
    count = len(os.listdir(cls_path))
    print(f"{cls}: {count} images")

print("✅ Step 3 completed: Hb filtering + class folders done.")


In [ ]:
#Resize_224_for_vision_transformer
import cv2
import os

# CONFIG
dataset_dir = "final_dataset"
target_size = (224, 224)  # width x height

# LOOP THROUGH CLASSES
for cls in os.listdir(dataset_dir):
    cls_path = os.path.join(dataset_dir, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        
        # READ IMAGE
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read image: {img_path}")
            continue
        
        # RESIZE
        img_resized = cv2.resize(img, target_size)
        
        # OVERWRITE ORIGINAL
        cv2.imwrite(img_path, img_resized)

print("✅ Step 4 completed: All images resized to 224x224.")


✅ Step 4 completed: All images resized to 224x224.


In [ ]:
#dataset_balance_check
import os

dataset_dir = "final_dataset"  # your folder with anemia, no_anemia, borderline

print("📊 Dataset Balance Check:")

for cls in os.listdir(dataset_dir):
    cls_path = os.path.join(dataset_dir, cls)
    count = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png'))])
    print(f"{cls}: {count} images")


📊 Dataset Balance Check:
no_anemia: 306 images
anemia: 368 images
10-13: 1574 images


In [ ]:
#data_splitting
import os
import shutil
import random

# -----------------------------
# CONFIG
# -----------------------------
dataset_dir = "final_dataset"  # your source folder with anemia/no_anemia/borderline
output_dir = "data"            # output folder for train/test
train_ratio = 0.8              # 80% train, 20% test

# -----------------------------
# CREATE FOLDER STRUCTURE
# -----------------------------
classes = ["anemia", "no_anemia"]  # we’ll skip borderline for training
for split in ["train", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

# -----------------------------
# SPLIT AND COPY IMAGES
# -----------------------------
for cls in classes:
    cls_path = os.path.join(dataset_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png'))]
    random.shuffle(images)  # shuffle before splitting

    train_count = int(len(images) * train_ratio)
    train_imgs = images[:train_count]
    test_imgs = images[train_count:]

    # COPY TRAIN IMAGES
    for img in train_imgs:
        src = os.path.join(cls_path, img)
        dst = os.path.join(output_dir, "train", cls, img)
        shutil.copy(src, dst)

    # COPY TEST IMAGES
    for img in test_imgs:
        src = os.path.join(cls_path, img)
        dst = os.path.join(output_dir, "test", cls, img)
        shutil.copy(src, dst)

    print(f"{cls}: {len(train_imgs)} train, {len(test_imgs)} test images")

print("✅ Step 6 completed: Train/Test split done.")


anemia: 294 train, 74 test images
no_anemia: 244 train, 62 test images
✅ Step 6 completed: Train/Test split done.
